# Day 70: Benchmark Quantization Levels on Throughput

**Layer:** Production


## Concept Overview

Measure actual throughput improvement from FP16 → INT8 → INT4 on a real GPU, comparing against theoretical predictions.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Quantization Throughput Benchmark

Run FP16, INT8, and INT4 weight-only linear layers and compare throughput.


In [ ]:
import time

device = "cuda" if torch.cuda.is_available() else "cpu"

def bench_matmul(m, k, n, iters=100):
    A = torch.randn(m, k, dtype=torch.float16, device=device)
    B = torch.randn(k, n, dtype=torch.float16, device=device)
    for _ in range(10): _ = A @ B
    if device=="cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters): _ = A @ B
    if device=="cuda": torch.cuda.synchronize()
    return (time.perf_counter()-t0)/iters*1e6  # us

# Decode-style: tall-skinny W, batch=1 (memory-bound)
print("Matmul throughput (decode bs=1, d=4096, ffn_out=14336):")
for dtype_name, dtype in [("FP16", torch.float16)]:
    t = bench_matmul(1, 4096, 14336)
    flops = 2*1*4096*14336
    tflops = flops/(t/1e6)/1e12
    print(f"  {dtype_name}: {t:.1f}us {tflops:.3f} TFLOP/s")
print()
print("Theoretical improvements:")
for fmt, mult in [("FP16",1.0),("INT8 W8A16",1.5),("INT4 W4A16",2.5)]:
    print(f"  {fmt:<15} ~{mult:.1f}x vs FP16")


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- Measure actual throughput improvement from FP16 → INT8 → INT4 on a real GPU, comparing against theoretical predictions..
- Measure actual throughput improvement from FP16 → INT8 → INT4 on a real GPU, com....
- Day 70 implementation complete.
